In [1]:
# TAHAP 1A IMPORT LIBRARY

print("Mulai mengimpor library yang dibutuhkan...")

import requests
import pandas as pd
from io import StringIO

print("Seluruh library berhasil diimpor.")

Mulai mengimpor library yang dibutuhkan...
Seluruh library berhasil diimpor.


In [2]:
# TAHAP 1B KONFIGURASI API E-STAT

print("Mulai menyiapkan konfigurasi API e-Stat...")

# Konfigurasi API
APP_ID = "69fbbb74fa35bfa86664ac12004c7b8b7296ea15"
STATS_DATA_ID = "0000010206"

print("Konfigurasi API berhasil disiapkan.")
print(f"Stats Data ID : {STATS_DATA_ID}")

Mulai menyiapkan konfigurasi API e-Stat...
Konfigurasi API berhasil disiapkan.
Stats Data ID : 0000010206


In [3]:
# TAHAP 1C PEMBENTUKAN URL REQUEST API

print("Mulai membentuk URL request API...")

# Membentuk URL API e-Stat
url = f"""
http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData
?appId={APP_ID}
&lang=E
&statsDataId={STATS_DATA_ID}
&metaGetFlg=Y
&cntGetFlg=N
&explanationGetFlg=Y
&annotationGetFlg=Y
&sectionHeaderFlg=1
&replaceSpChars=0
"""

# Membersihkan format URL
url = url.replace("\n", "").replace(" ", "")

print("URL request API berhasil dibuat.")

Mulai membentuk URL request API...
URL request API berhasil dibuat.


In [4]:
# TAHAP 1D PENGIRIMAN REQUEST KE API

print("Mulai menghubungkan sistem ke API e-Stat...")

# Mengirim request ke API
response = requests.get(url)

print("Request API berhasil dikirim.")
print(f"Status koneksi API : {response.status_code}")

Mulai menghubungkan sistem ke API e-Stat...
Request API berhasil dikirim.
Status koneksi API : 200


In [5]:
# TAHAP 1E PEMBACAAN STRUKTUR DATA RESPON API

print("Mulai membaca struktur data dari API...")

# Memisahkan isi response menjadi baris teks
lines = response.text.splitlines()

# Mencari posisi awal tabel data
value_index = None

for i, line in enumerate(lines):
    if line.strip().replace('"', '') == "VALUE":
        value_index = i
        break

if value_index is not None:
    print("Penanda awal data berhasil ditemukan.")
    print(f"Posisi baris VALUE ditemukan pada index ke-{value_index}")
else:
    print("Penanda VALUE tidak ditemukan.")

Mulai membaca struktur data dari API...
Penanda awal data berhasil ditemukan.
Posisi baris VALUE ditemukan pada index ke-27


In [6]:
APP_ID = "69fbbb74fa35bfa86664ac12004c7b8b7296ea15"
STATS_DATA_ID = "0004007161"

url = f"""
http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData
?appId={APP_ID}
&lang=J
&statsDataId={STATS_DATA_ID}
&metaGetFlg=Y
&cntGetFlg=N
&explanationGetFlg=Y
&annotationGetFlg=Y
&sectionHeaderFlg=1
&replaceSpChars=0
"""

url = url.replace("\n", "").replace(" ", "")

response = requests.get(url)

print(f"Status koneksi API: {response.status_code}")

if response.status_code == 200:
    print("Koneksi API berhasil.")
else:
    print("Koneksi API gagal.")

Status koneksi API: 200
Koneksi API berhasil.


In [7]:
# TAHAP 1F PEMBACAAN DATA KE DATAFRAME

print("Mulai memuat data ke dalam DataFrame...")

if value_index is not None:

    # Mengambil isi tabel data
    csv_text = "\n".join(lines[value_index + 1:])

    # Membaca data ke DataFrame
    df_kerja_mentah = pd.read_csv(
        StringIO(csv_text)
    )

    # --- TAMBAHAN PATH TARGET UNTUK DATA MENTAH LOWONGAN KERJA ---
    path_raw_kerja = "../data/raw/raw_kondisi_kerja_japan.csv"

    # Mengekspor data mentah asli dari API ke folder raw
    df_kerja_mentah.to_csv(path_raw_kerja, index=False, encoding='utf-8')
    # -------------------------------------------------------------

    print("Data berhasil dimuat ke DataFrame dan disimpan fisik.")
    print(f"Lokasi penyimpanan mentah: {path_raw_kerja}")

else:
    print("Data tidak dapat dimuat karena VALUE tidak ditemukan.")

Mulai memuat data ke dalam DataFrame...
Data berhasil dimuat ke DataFrame dan disimpan fisik.
Lokasi penyimpanan mentah: ../data/raw/raw_kondisi_kerja_japan.csv


In [8]:
# TAHAP 1G VERIFIKASI HASIL DATA INGESTION

print("\n=============================================")
print("         TAHAP 1 SELESAI (KONDISI KERJA)     ")
print("=============================================")

print("Pengambilan data mentah berhasil dilakukan.")
print(f"Total data awal : {df_kerja_mentah.shape[0]} baris")
print(f"Total kolom     : {df_kerja_mentah.shape[1]} kolom")

print("\n--- Daftar Kolom Dataset ---")

print(df_kerja_mentah.columns.tolist())


         TAHAP 1 SELESAI (KONDISI KERJA)     
Pengambilan data mentah berhasil dilakukan.
Total data awal : 89472 baris
Total kolom     : 11 kolom

--- Daftar Kolom Dataset ---
['tab_code', 'Observation Value', 'cat01_code', 'F Labour', 'area_code', 'AREA', 'time_code', 'SURVEY YEAR', 'unit', 'value', 'annotation']


In [9]:
# TAHAP 2A PERSIAPAN DATA FILTERING

print("Mulai menyiapkan data untuk proses filtering...")

# Membuat salinan data mentah
df_kerja_filter = df_kerja_mentah.copy()

print("Data berhasil disiapkan.")
print(f"Total data awal : {df_kerja_filter.shape[0]} baris")

Mulai menyiapkan data untuk proses filtering...
Data berhasil disiapkan.
Total data awal : 89472 baris


In [10]:
# TAHAP 2B FILTER DATA NASIONAL

print("Mulai menghapus data nasional All Japan...")

# Menyisakan data prefektur saja
df_kerja_filter = (
    df_kerja_filter[
        df_kerja_filter['AREA'] != 'All Japan'
    ]
)

print("Data nasional berhasil dihapus.")
print(f"Total data saat ini : {df_kerja_filter.shape[0]} baris")

Mulai menghapus data nasional All Japan...
Data nasional berhasil dihapus.
Total data saat ini : 87608 baris


In [11]:
# TAHAP 2C FILTER INDIKATOR RASIO LOWONGAN KERJA

print("Mulai memfilter indikator rasio lowongan kerja...")

# Kode indikator rasio lowongan kerja
KODE_RASIO_KERJA = "#F03105"

# Menyaring indikator
df_kerja_filter = (
    df_kerja_filter[
        df_kerja_filter['cat01_code'] == KODE_RASIO_KERJA
    ]
)

print("Filter indikator berhasil dilakukan.")
print(f"Indikator yang digunakan : {KODE_RASIO_KERJA}")

Mulai memfilter indikator rasio lowongan kerja...
Filter indikator berhasil dilakukan.
Indikator yang digunakan : #F03105


In [12]:
# TAHAP 2D KONVERSI FORMAT TAHUN

print("Mulai mengubah format data tahun...")

# Mengubah tipe data tahun menjadi integer
df_kerja_filter['SURVEY YEAR'] = (
    df_kerja_filter['SURVEY YEAR']
    .astype(int)
)

print("Konversi format tahun berhasil dilakukan.")

Mulai mengubah format data tahun...
Konversi format tahun berhasil dilakukan.


In [13]:
# TAHAP 2E FILTER RENTANG TAHUN PENELITIAN

print("Mulai memfilter rentang tahun penelitian...")

# Menentukan rentang tahun penelitian
tahun_target_kerja = [
    2020,
    2021,
    2022,
    2023,
    2024
]

# Menyaring data berdasarkan tahun
df_kerja_filter = (
    df_kerja_filter[
        df_kerja_filter['SURVEY YEAR']
        .isin(tahun_target_kerja)
    ]
)

print("Filter rentang tahun berhasil dilakukan.")

Mulai memfilter rentang tahun penelitian...
Filter rentang tahun berhasil dilakukan.


In [14]:
# TAHAP 2F VERIFIKASI HASIL FILTERING

print("\n=============================================")
print("         TAHAP 2 SELESAI (KONDISI KERJA)     ")
print("=============================================")

print("Penyaringan data rasio lowongan kerja berhasil dilakukan.")
print(f"Total data hasil filtering : {df_kerja_filter.shape[0]} baris")

print("\nTahun yang terdeteksi pada dataset :")
print(sorted(df_kerja_filter['SURVEY YEAR'].unique()))

print("\n--- Sampel Hasil Filtering ---")

kolom_tampil = [
    'AREA',
    'SURVEY YEAR',
    'unit',
    'value'
]

print(df_kerja_filter[kolom_tampil].head(5))


         TAHAP 2 SELESAI (KONDISI KERJA)     
Penyaringan data rasio lowongan kerja berhasil dilakukan.
Total data hasil filtering : 235 baris

Tahun yang terdeteksi pada dataset :
[np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

--- Sampel Hasil Filtering ---
           AREA  SURVEY YEAR   unit value
14751  Hokkaido         2020  times  0.98
14752  Hokkaido         2021  times  1.03
14753  Hokkaido         2022  times  1.15
14754  Hokkaido         2023  times  1.04
14755  Hokkaido         2024  times  0.99


In [15]:
import numpy as np

# TAHAP 3A SALINAN DATA HASIL FILTER

print("Mulai melakukan pembersihan data kosong dan konversi tipe data...")

# Mengambil salinan data hasil penyaringan dari Tahap 2
df_kerja_clean = df_kerja_filter.copy()

print("Salinan data berhasil dibuat.")
print(f"Total data awal: {df_kerja_clean.shape[0]} baris")

Mulai melakukan pembersihan data kosong dan konversi tipe data...
Salinan data berhasil dibuat.
Total data awal: 235 baris


In [16]:
# TAHAP 3B PEMBERSIHAN FORMAT NILAI KOLOM VALUE

# Menghapus spasi tersembunyi dan mengubah tipe data sementara menjadi string
df_kerja_clean['value'] = df_kerja_clean['value'].astype(str).str.strip()

# Mengganti simbol data kosong menjadi NaN
df_kerja_clean['value'] = df_kerja_clean['value'].replace(
    ['-', '‐', ''],
    np.nan
)

print("Format nilai pada kolom 'value' berhasil dibersihkan.")

Format nilai pada kolom 'value' berhasil dibersihkan.


In [17]:
# TAHAP 3C PENGHAPUSAN DATA KOSONG

# Menghapus baris yang memiliki nilai kosong pada kolom value
df_kerja_clean = df_kerja_clean.dropna(subset=['value'])

print("Baris dengan nilai kosong berhasil dihapus.")
print(f"Total data setelah pembersihan: {df_kerja_clean.shape[0]} baris")

Baris dengan nilai kosong berhasil dihapus.
Total data setelah pembersihan: 235 baris


In [18]:
# TAHAP 3D KONVERSI TIPE DATA NUMERIK

# Mengubah kolom value menjadi tipe float
df_kerja_clean['value'] = df_kerja_clean['value'].astype(float)

print("Konversi tipe data numerik berhasil dilakukan.")

Konversi tipe data numerik berhasil dilakukan.


In [19]:
# TAHAP 3E VERIFIKASI HASIL PEMBERSIHAN DATA

print("\n=============================================")
print("         TAHAP 3 SELESAI (KONDISI KERJA)     ")
print("=============================================")

print("Pembersihan data dan konversi tipe data berhasil dilakukan.")
print(f"Total data akhir: {df_kerja_clean.shape[0]} baris")

print("\n--- Verifikasi Tipe Data Kolom 'value' ---")
print(df_kerja_clean[['value']].info())


         TAHAP 3 SELESAI (KONDISI KERJA)     
Pembersihan data dan konversi tipe data berhasil dilakukan.
Total data akhir: 235 baris

--- Verifikasi Tipe Data Kolom 'value' ---
<class 'pandas.core.frame.DataFrame'>
Index: 235 entries, 14751 to 15215
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   value   235 non-null    float64
dtypes: float64(1)
memory usage: 3.7 KB
None


In [20]:
# TAHAP 4A SALINAN DATA HASIL PEMBERSIHAN

print("Mulai membersihkan dan menstandardisasi nama prefektur...")

# Mengambil salinan data bersih dari Tahap 3
df_kerja_standard = df_kerja_clean.copy()

print("Salinan data berhasil dibuat.")
print(f"Total data awal: {df_kerja_standard.shape[0]} baris")

Mulai membersihkan dan menstandardisasi nama prefektur...
Salinan data berhasil dibuat.
Total data awal: 235 baris


In [21]:
# TAHAP 4B PEMBERSIHAN SUFFIX NAMA PREFEKTUR

# Menghapus suffix administratif Jepang pada nama prefektur
df_kerja_standard['AREA'] = df_kerja_standard['AREA'].str.replace(
    '-ken',
    '',
    case=False,
    regex=False
)

df_kerja_standard['AREA'] = df_kerja_standard['AREA'].str.replace(
    '-to',
    '',
    case=False,
    regex=False
)

df_kerja_standard['AREA'] = df_kerja_standard['AREA'].str.replace(
    '-fu',
    '',
    case=False,
    regex=False
)

print("Suffix administratif prefektur berhasil dibersihkan.")

Suffix administratif prefektur berhasil dibersihkan.


In [22]:
# TAHAP 4C PEMBERSIHAN SPASI TEKS

# Menghapus spasi berlebih pada awal dan akhir teks
df_kerja_standard['AREA'] = df_kerja_standard['AREA'].str.strip()

print("Spasi berlebih berhasil dibersihkan.")

Spasi berlebih berhasil dibersihkan.


In [23]:
# TAHAP 4D PENYERAGAMAN NAMA KOLOM

# Mengubah nama kolom AREA menjadi prefektur_en
df_kerja_standard = df_kerja_standard.rename(
    columns={'AREA': 'prefektur_en'}
)

print("Nama kolom berhasil diseragamkan.")

Nama kolom berhasil diseragamkan.


In [24]:
# TAHAP 4E VERIFIKASI HASIL STANDARISASI

print("\n=============================================")
print("         TAHAP 4 SELESAI (KONDISI KERJA)     ")
print("=============================================")

print("Standarisasi nama prefektur berhasil dilakukan.")
print(f"Total data saat ini: {df_kerja_standard.shape[0]} baris")

print("\n--- Sampel 5 Data Pertama Hasil Standarisasi Nama ---")

kolom_tampil = [
    'prefektur_en',
    'SURVEY YEAR',
    'value'
]

print(df_kerja_standard[kolom_tampil].head())


         TAHAP 4 SELESAI (KONDISI KERJA)     
Standarisasi nama prefektur berhasil dilakukan.
Total data saat ini: 235 baris

--- Sampel 5 Data Pertama Hasil Standarisasi Nama ---
      prefektur_en  SURVEY YEAR  value
14751     Hokkaido         2020   0.98
14752     Hokkaido         2021   1.03
14753     Hokkaido         2022   1.15
14754     Hokkaido         2023   1.04
14755     Hokkaido         2024   0.99


In [25]:
# TAHAP 5A SALINAN DATA HASIL STANDARISASI

print("Mulai menambahkan metadata dan pelabelan indikator sistem...")

# Mengambil salinan data hasil standarisasi
df_kerja_label = df_kerja_standard.copy()

print("Salinan data berhasil dibuat.")
print(f"Total data awal: {df_kerja_label.shape[0]} baris")

Mulai menambahkan metadata dan pelabelan indikator sistem...
Salinan data berhasil dibuat.
Total data awal: 235 baris


In [26]:
# TAHAP 5B PENAMBAHAN KATEGORI KRITERIA

# Menambahkan kategori induk untuk kebutuhan sistem SPK
df_kerja_label['kategori_kondisi_id'] = 'Kondisi Ketenagakerjaan'

print("Kolom kategori_kondisi_id berhasil ditambahkan.")

Kolom kategori_kondisi_id berhasil ditambahkan.


In [27]:
# TAHAP 5C PENAMBAHAN NAMA INDIKATOR

# Menambahkan nama indikator spesifik
df_kerja_label['indikator_id'] = 'Rasio Lowongan Kerja'

print("Kolom indikator_id berhasil ditambahkan.")

Kolom indikator_id berhasil ditambahkan.


In [28]:
# TAHAP 5D VERIFIKASI HASIL PELABELAN

print("\n=============================================")
print("         TAHAP 5 SELESAI (KONDISI KERJA)     ")
print("=============================================")

print("Metadata dan pelabelan indikator berhasil ditambahkan.")
print(f"Total data saat ini: {df_kerja_label.shape[0]} baris")

print("\n--- Sampel 5 Data Pertama Hasil Pelabelan Metadata ---")

kolom_tampil = [
    'prefektur_en',
    'SURVEY YEAR',
    'kategori_kondisi_id',
    'indikator_id',
    'value'
]

print(df_kerja_label[kolom_tampil].head())


         TAHAP 5 SELESAI (KONDISI KERJA)     
Metadata dan pelabelan indikator berhasil ditambahkan.
Total data saat ini: 235 baris

--- Sampel 5 Data Pertama Hasil Pelabelan Metadata ---
      prefektur_en  SURVEY YEAR      kategori_kondisi_id  \
14751     Hokkaido         2020  Kondisi Ketenagakerjaan   
14752     Hokkaido         2021  Kondisi Ketenagakerjaan   
14753     Hokkaido         2022  Kondisi Ketenagakerjaan   
14754     Hokkaido         2023  Kondisi Ketenagakerjaan   
14755     Hokkaido         2024  Kondisi Ketenagakerjaan   

               indikator_id  value  
14751  Rasio Lowongan Kerja   0.98  
14752  Rasio Lowongan Kerja   1.03  
14753  Rasio Lowongan Kerja   1.15  
14754  Rasio Lowongan Kerja   1.04  
14755  Rasio Lowongan Kerja   0.99  


In [29]:
# TAHAP 6A SALINAN DATA HASIL PELABELAN

print("Mulai menambahkan atribut kriteria SAW...")

# Mengambil salinan data hasil pelabelan
df_kerja_saw = df_kerja_label.copy()

print("Salinan data berhasil dibuat.")
print(f"Total data awal: {df_kerja_saw.shape[0]} baris")

Mulai menambahkan atribut kriteria SAW...
Salinan data berhasil dibuat.
Total data awal: 235 baris


In [30]:
# TAHAP 6B PENAMBAHAN ATRIBUT KRITERIA SAW

# Menambahkan atribut tipe kriteria SAW
df_kerja_saw['tipe_kriteria_kerja'] = 'Benefit'

print("Kolom tipe_kriteria_kerja berhasil ditambahkan.")

Kolom tipe_kriteria_kerja berhasil ditambahkan.


In [31]:
# TAHAP 6C VERIFIKASI HASIL ATRIBUT SAW

print("\n=============================================")
print("         TAHAP 6 SELESAI (KONDISI KERJA)     ")
print("=============================================")

print("Penambahan atribut SAW berhasil dilakukan.")
print(f"Total data saat ini: {df_kerja_saw.shape[0]} baris")

print("\n--- Sampel 5 Data Pertama Hasil Atribut SAW ---")

kolom_tampil = [
    'prefektur_en',
    'SURVEY YEAR',
    'value',
    'tipe_kriteria_kerja'
]

print(df_kerja_saw[kolom_tampil].head())


         TAHAP 6 SELESAI (KONDISI KERJA)     
Penambahan atribut SAW berhasil dilakukan.
Total data saat ini: 235 baris

--- Sampel 5 Data Pertama Hasil Atribut SAW ---
      prefektur_en  SURVEY YEAR  value tipe_kriteria_kerja
14751     Hokkaido         2020   0.98             Benefit
14752     Hokkaido         2021   1.03             Benefit
14753     Hokkaido         2022   1.15             Benefit
14754     Hokkaido         2023   1.04             Benefit
14755     Hokkaido         2024   0.99             Benefit


In [32]:
# TAHAP 7A PERSIAPAN AGREGASI DATA

print("Mulai menghitung rata-rata rasio lowongan kerja per prefektur...")

# Mengambil salinan data hasil atribut SAW
df_kerja_final = df_kerja_saw.copy()

print("Data agregasi berhasil disiapkan.")
print(f"Total data awal: {df_kerja_final.shape[0]} baris")

Mulai menghitung rata-rata rasio lowongan kerja per prefektur...
Data agregasi berhasil disiapkan.
Total data awal: 235 baris


In [33]:
# TAHAP 7B AGREGASI RATA-RATA RASIO LOWONGAN KERJA

# Mengelompokkan data berdasarkan prefektur dan metadata
df_kerja_final = df_kerja_final.groupby(
    [
        'prefektur_en',
        'kategori_kondisi_id',
        'indikator_id',
        'tipe_kriteria_kerja'
    ],
    as_index=False
).agg({
    'value': 'mean'
})

print("Agregasi rata-rata berhasil dilakukan.")

Agregasi rata-rata berhasil dilakukan.


In [34]:
# TAHAP 7C PEMBENTUKAN KOLOM HASIL AKHIR

# Membentuk kolom rasio lowongan kerja hasil agregasi
df_kerja_final['rasio_lowongan_kerja'] = (
    df_kerja_final['value']
    .round(2)
)

# Menghapus kolom value lama
df_kerja_final = df_kerja_final.drop(columns=['value'])

print("Kolom hasil akhir berhasil dibentuk.")

Kolom hasil akhir berhasil dibentuk.


In [35]:
# TAHAP 7D PENGURUTAN DATA FINAL

# Mengurutkan data berdasarkan nama prefektur
df_kerja_final = (
    df_kerja_final
    .sort_values(by=['prefektur_en'])
    .reset_index(drop=True)
)

print("Data final berhasil diurutkan.")

Data final berhasil diurutkan.


In [36]:
# TAHAP 7E EKSPOR DATA FINAL KE CSV

print("Mulai mengekspor dataset final kondisi kerja...")

# --- PERUBAHAN PATH TARGET UNTUK DATA FINAL KONDISI KERJA ---
nama_file_final = "../data/processed/dataset_kondisi_kerja_final.csv"

# Mengekspor dataset ke file CSV
df_kerja_final.to_csv(
    nama_file_final,
    index=False,
    encoding='utf-8'
)

print("File CSV final berhasil disimpan.")
print(f"Nama file / Jalur: {nama_file_final}")

Mulai mengekspor dataset final kondisi kerja...
File CSV final berhasil disimpan.
Nama file / Jalur: ../data/processed/dataset_kondisi_kerja_final.csv


In [37]:
# TAHAP 7F VERIFIKASI HASIL DATA FINAL

print("\n=============================================")
print("     ALL TAHAPAN SUKSES & DATASET SELESAI    ")
print("=============================================")

print("Dataset kondisi kerja berhasil diproses.")
print(f"Total data final SPK: {df_kerja_final.shape[0]} baris")

print("\n--- Sampel 5 Data Teratas Dataset Final ---")

print(df_kerja_final.head())


     ALL TAHAPAN SUKSES & DATASET SELESAI    
Dataset kondisi kerja berhasil diproses.
Total data final SPK: 47 baris

--- Sampel 5 Data Teratas Dataset Final ---
  prefektur_en      kategori_kondisi_id          indikator_id  \
0        Aichi  Kondisi Ketenagakerjaan  Rasio Lowongan Kerja   
1        Akita  Kondisi Ketenagakerjaan  Rasio Lowongan Kerja   
2       Aomori  Kondisi Ketenagakerjaan  Rasio Lowongan Kerja   
3        Chiba  Kondisi Ketenagakerjaan  Rasio Lowongan Kerja   
4        Ehime  Kondisi Ketenagakerjaan  Rasio Lowongan Kerja   

  tipe_kriteria_kerja  rasio_lowongan_kerja  
0             Benefit                  1.27  
1             Benefit                  1.37  
2             Benefit                  1.10  
3             Benefit                  0.95  
4             Benefit                  1.35  


In [38]:
df_kerja_final.columns.tolist()

['prefektur_en',
 'kategori_kondisi_id',
 'indikator_id',
 'tipe_kriteria_kerja',
 'rasio_lowongan_kerja']